# Canwell Glacier: sites, domain, and HRRR-AK grid

This notebook maps Pascal Buri's five sites, the supplied **EPSG:32606** area of interest,
and the buffered geographic window used by the active Canwell download. It opens one
local GRIB2 file and plots **2 m air temperature (°C)** with the native cell boundaries.
A separate satellite map contains no HRRR temperature data.

Site latitude/longitude coordinates come directly from `data/canwell_readme.txt`.
The supplied UTM extent defines the area of interest; individual UTM station coordinates
are preserved in that source file. GRIB files remain in the HRRR archive outside this repo.
Run all cells from this repo or its `notebooks` directory. The first imagery request needs
internet access; subsequent runs use a local cache.

In [ ]:
from pathlib import Path
import csv
import io
import json
import os
import re
import shutil
import subprocess
import tempfile
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon
from PIL import Image
from pyproj import CRS, Transformer

# Find the repo whether Jupyter starts in its root or notebooks directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data' / 'canwell_readme.txt').exists())
FIGURES = ROOT / 'figures'
CACHE = ROOT / 'data' / 'basemaps'
FIGURES.mkdir(exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

# Override CANWELL_GRIB to inspect another completed file.
GRIB = Path(os.environ.get('CANWELL_GRIB',
    '/hdd/snow_hydrology/hrrrak/canwell_domain/f567/WY2020/'
    'hrrr.20190801/hrrr.t00z.wrfsfcf05.ak.grib2'))
WGRIB2 = shutil.which('wgrib2') or '/usr/local/bin/wgrib2'
assert GRIB.is_file(), f'GRIB not found: {GRIB}; set CANWELL_GRIB to a downloaded file.'
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print(f'Repository: {ROOT}\nGRIB: {GRIB}')

## Source coordinates and download window

The 12 km buffer is applied in UTM before converting to longitude/latitude. The downloader
uses the enclosing longitude/latitude rectangle, not the original UTM rectangle. Subsetting
keeps whole native grid cells, so the resulting GRIB footprint differs from that window.

In [ ]:
source = (ROOT / 'data' / 'canwell_readme.txt').read_text()
crs_code = int(re.search(r'CRS\): EPSG:(\d+)', source).group(1))
utm = CRS.from_epsg(crs_code)
wgs84 = CRS.from_epsg(4326)
to_utm = Transformer.from_crs(wgs84, utm, always_xy=True)
to_lonlat = Transformer.from_crs(utm, wgs84, always_xy=True)
aoi = tuple(map(float, re.search(
    r'Extent \(xmin,ymin,xmax,ymax\): ([0-9, ]+)', source).group(1).split(',')))
sites = []
for block in source.split('### '):
    match = re.search(r'DD: ([0-9.]+),\s*(-[0-9.]+)', block)
    if match:
        name = block.split(' ###')[0].strip()
        lat, lon = map(float, match.groups())
        elev = float(re.search(r'Elev\.: ([0-9.]+)m', block).group(1))
        x, y = to_utm.transform(lon, lat)
        sites.append(dict(name=name, lat=lat, lon=lon, elevation_m=elev, x=x, y=y))
assert len(sites) == 5
buffer_m = 12_000
buffered = (aoi[0]-buffer_m, aoi[1]-buffer_m, aoi[2]+buffer_m, aoi[3]+buffer_m)
# These are the outward-rounded bounds actually configured in the downloader.
window = (-145.915909, 63.134289, -145.015718, 63.493377)
computed = to_lonlat.transform_bounds(*buffered, densify_pts=101)
assert np.max(np.abs(np.asarray(window)-computed)) < 1e-6
print('AOI UTM:', aoi)
print('Download window (west, south, east, north):', window)
for i, site in enumerate(sites, 1):
    print(f"{i}. {site['name']}: {site['lat']:.5f}, {site['lon']:.5f}; {site['elevation_m']:.0f} m")

# Densify each rectangle edge before changing projection.
def rectangle(bounds, n=101):
    west, south, east, north = bounds
    return np.array([(x, south) for x in np.linspace(west, east, n)] +
                    [(east, y) for y in np.linspace(south, north, n)] +
                    [(x, north) for x in np.linspace(east, west, n)] +
                    [(west, y) for y in np.linspace(north, south, n)])

aoi_ring = rectangle(aoi)
window_ring = np.column_stack(to_utm.transform(*rectangle(window).T))

## Open one GRIB2 file

[`wgrib2 -csv`](https://www.cpc.ncep.noaa.gov/products/wesley/wgrib2/csv.html)
decodes only `TMP:2 m above ground`. Its CSV contains initialization time, valid time,
longitude, latitude, and temperature in kelvin. This avoids needing a separate GRIB Python
backend. Native projection parameters, dimensions, and Earth radius are read from the file.
Cell corners are constructed at half-grid offsets in the native projection, then transformed
to UTM; this preserves the cell spacing and orientation.

In [ ]:
match = ':TMP:2 m above ground:'
env = dict(os.environ, OMP_NUM_THREADS='1')
def wgrib(*args):
    return subprocess.run([WGRIB2, str(GRIB), '-match', match, *args],
                          check=True, text=True, capture_output=True, env=env).stdout
inventory = wgrib('-s')
assert len(inventory.strip().splitlines()) == 1, 'Expected exactly one 2 m temperature field.'
metadata = wgrib('-grid', '-radius')
print(inventory.strip())
print(metadata.strip())
with tempfile.TemporaryDirectory() as temporary:
    csv_path = Path(temporary) / 'temperature.csv'
    wgrib('-order', 'we:sn', '-csv', str(csv_path))
    rows = list(csv.reader(io.StringIO(csv_path.read_text())))

nx, ny = map(int, re.search(r'\((\d+) x (\d+)\)', metadata).groups())
assert len(rows) == nx * ny, 'Unexpected missing or extra grid points.'
assert all(row[2:4] == ['TMP', '2 m above ground'] for row in rows)
values = np.array([list(map(float, row[4:7])) for row in rows])
lon = ((values[:, 0] + 180) % 360 - 180).reshape(ny, nx)
lat = values[:, 1].reshape(ny, nx)
temperature = values[:, 2].reshape(ny, nx)
temperature = np.ma.masked_where(np.abs(temperature) > 1e10, temperature) - 273.15
initial_time, valid_time = rows[0][:2]

def parameter(name):
    return float(re.search(rf'{name} ([0-9.+-]+)', metadata).group(1))
assert 'polar stereographic' in metadata and 'North pole' in metadata
assert 'is_spherical=1' in metadata
radius = float(re.search(r'radius_maj=([0-9.]+)', metadata).group(1))
native = CRS.from_proj4(f'+proj=stere +lat_0=90 +lat_ts={parameter("latD")} '
                       f'+lon_0={parameter("lonV")} +R={radius} +units=m')
to_native = Transformer.from_crs(wgs84, native, always_xy=True)
native_to_utm = Transformer.from_crs(native, utm, always_xy=True)
x0, y0 = to_native.transform(parameter('lon1'), parameter('lat1'))
dx, dy = parameter('dx'), parameter('dy')
xc, yc = np.meshgrid(x0 + np.arange(nx)*dx, y0 + np.arange(ny)*dy)
# Compare native reconstruction with the rounded CSV coordinates.
raw_x, raw_y = to_native.transform(lon, lat)
assert np.max(np.hypot(raw_x-xc, raw_y-yc)) < 100, 'Grid scan orientation mismatch.'
xe, ye = np.meshgrid(x0 + (np.arange(nx+1)-0.5)*dx,
                     y0 + (np.arange(ny+1)-0.5)*dy)
grid_x, grid_y = native_to_utm.transform(xe, ye)
center_x, center_y = native_to_utm.transform(xc, yc)
footprint = np.concatenate([
    np.column_stack([grid_x[0, :], grid_y[0, :]]),
    np.column_stack([grid_x[:, -1], grid_y[:, -1]]),
    np.column_stack([grid_x[-1, ::-1], grid_y[-1, ::-1]]),
    np.column_stack([grid_x[::-1, 0], grid_y[::-1, 0]])])
print(f'{nx} × {ny} cells; native spacing {dx/1000:g} × {dy/1000:g} km')
print(f'Initialized {initial_time} UTC; valid {valid_time} UTC')
print(f'Temperature: {temperature.min():.1f} to {temperature.max():.1f} °C')

## Satellite basemap: sites and domain outlines

Imagery: [Esri World Imagery](https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer).
It provides regional context; its acquisition dates differ from the forecast date.
The image and returned georeferencing are cached under `data/basemaps/` (excluded from Git).
No temperature values appear in this map. Both maps use UTM zone 6N in kilometres.

In [ ]:
# Request a small regional image already projected to the map CRS.
margin = 2000
view = (min(grid_x.min(), window_ring[:, 0].min())-margin,
        min(grid_y.min(), window_ring[:, 1].min())-margin,
        max(grid_x.max(), window_ring[:, 0].max())+margin,
        max(grid_y.max(), window_ring[:, 1].max())+margin)
service = 'https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer'
params = dict(bbox=','.join(map(str, view)), bboxSR=crs_code, imageSR=crs_code,
              size='1400,1200', format='png', transparent='false', f='json')
import hashlib
cache_key = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:16]
image_path = CACHE / f'esri_{cache_key}.png'
metadata_path = CACHE / f'esri_{cache_key}.json'
def fetch(url):
    request = Request(url, headers={'User-Agent': 'CanwellGlacierResearch/1.0'})
    with urlopen(request, timeout=90) as response:
        return response.read()
if not image_path.exists() or not metadata_path.exists():
    info = json.loads(fetch(service + '/export?' + urlencode(params)))
    if 'error' in info:
        raise RuntimeError(f'Basemap service returned: {info["error"]}')
    image_bytes = fetch(info['href'])
    # Validate before writing a reusable cache.
    Image.open(io.BytesIO(image_bytes)).verify()
    image_path.write_bytes(image_bytes)
    metadata_path.write_text(json.dumps(info, indent=2))
info = json.loads(metadata_path.read_text())
image = np.asarray(Image.open(image_path).convert('RGB'))
b = info['extent']
image_extent = np.array([b['xmin'], b['xmax'], b['ymin'], b['ymax']]) / 1000

site_names = ['South Moraine', 'Look Eyrie AWS', 'Lower Canwell GPS',
              'Middle Canwell GPS', 'Upper Canwell / planned accumulation']
def overlays(ax, footprint_outline=False):
    ax.add_patch(Polygon(aoi_ring/1000, fill=False, edgecolor='#00ffff', lw=2.2,
                         label='Provided domain'))
    ax.add_patch(Polygon(window_ring/1000, fill=False, edgecolor='#ffb000', lw=2,
                         linestyle='--', label='HRRR download window'))
    if footprint_outline:
        ax.add_patch(Polygon(footprint/1000, fill=False, edgecolor='#ffffff', lw=1.4,
                             linestyle=':', label='Native subset cell footprint'))
    for i, site in enumerate(sites, 1):
        ax.scatter(site['x']/1000, site['y']/1000, s=65, c='white', edgecolors='black', zorder=5)
        ax.annotate(str(i), (site['x']/1000, site['y']/1000), xytext=(5, 7),
                    textcoords='offset points', color='black', weight='bold', zorder=6,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=1))
    ax.set(xlim=(view[0]/1000, view[2]/1000), ylim=(view[1]/1000, view[3]/1000),
           xlabel='UTM zone 6N easting (km)', ylabel='UTM zone 6N northing (km)')
    ax.set_aspect('equal')
    ax.ticklabel_format(useOffset=False, style='plain')
    ax.grid(alpha=0.15)
    ax.legend(loc='upper left', fontsize=8, framealpha=0.95)

def site_key(fig):
    fig.text(0.12, 0.045, 'Sites:  ' + '   |   '.join(
        f'{i}. {name}' for i, name in enumerate(site_names[:3], 1)) + '\n' +
        '           ' + '   |   '.join(f'{i}. {name}' for i, name in enumerate(site_names[3:], 4)),
        fontsize=8, va='bottom')

fig, ax = plt.subplots(figsize=(11, 11))
ax.imshow(image, extent=image_extent, origin='upper', zorder=0)
overlays(ax)
ax.set_title('Canwell Glacier: sites and domain extents\nSatellite imagery · no HRRR temperature overlay')
site_key(fig)
fig.text(0.12, 0.015, 'Imagery sources: Esri, Vantor, Earthstar Geographics, and the GIS User Community', fontsize=7)
fig.subplots_adjust(bottom=0.13, top=0.92)
fig.savefig(FIGURES / 'canwell_sites_satellite.png', dpi=180, bbox_inches='tight')
plt.show()

## HRRR-AK 2 m air temperature and native grid

Each quadrilateral is one native HRRR grid cell. Dots mark cell centres. The cyan outline
is the supplied domain; the orange dashed line is the geographic download window.
Native cells extend beyond that rectangle because subsetting preserves complete grid rows
and columns. These are forecast temperatures for a single timestamp, not station observations.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 11))
ax.set_facecolor('#464e56')
mesh = ax.pcolormesh(grid_x/1000, grid_y/1000, temperature, shading='flat',
                     cmap='coolwarm', edgecolors=(0, 0, 0, 0.45), linewidth=0.45,
                     rasterized=True)
ax.scatter(center_x/1000, center_y/1000, s=3, c='black', alpha=0.45)
overlays(ax, footprint_outline=True)
colorbar = fig.colorbar(mesh, ax=ax, shrink=0.74, pad=0.025)
colorbar.set_label('2 m air temperature (°C)')
ax.set_title(f'Canwell Glacier: HRRR-AK 2 m air temperature\n'
             f'Valid {valid_time} UTC · {nx} × {ny} cells · native {dx/1000:g} km spacing')
site_key(fig)
fig.text(0.12, 0.015, f'NOAA HRRR-AK · initialized {initial_time} UTC · {GRIB.name}', fontsize=7)
fig.subplots_adjust(bottom=0.13, top=0.92)
fig.savefig(FIGURES / 'canwell_hrrr_temperature_grid.png', dpi=180, bbox_inches='tight')
plt.show()

## Reproducibility notes

- Change `CANWELL_GRIB` (or `GRIB` in the first cell) to inspect another completed forecast.
- All model timestamps are UTC. Site elevations are source metadata and do not alter the plot.
- Map axes use WGS 84 / UTM zone 6N. Grid corners use the spherical polar stereographic
  parameters encoded in the GRIB before transforming to the plotting CRS.
- The download window includes a nominal four-cell buffer on each UTM side before conversion.
  Its geographic bounding box and the native extracted footprint are distinct outlines.
- The notebook reads the GRIB without modifying it or creating index files in the archive.
- Only this single small GRIB is decoded; the running download is unaffected.
- PNG exports are written to `figures/`. The notebook retains its executed plots.
